In [1]:
import os
import pandas as pd
from maomao.parsing.parsing_utils import *
from maomao.utils.constants import *

#### Processing and standardizing peptide datasets (HemoPI2.0)

This notebook curates the **HemoPI 2.0** dataset by integrating the two official components provided by the authors: a cross-validation dataset and an independent test dataset. The sequences and labels are standardized, duplicate consistency checks are applied, and the final curated dataset and metadata are exported for downstream analysis.

- **Toxic effect / endpoint:** hemolytic
- **Source:** HemoPI2.0
- **Sequence scope:** only non-modified peptide sequences are retained for the final dataset.

The pipeline performs the following steps:

- **Loads the two HemoPI 2.0 datasets**:
  - `cross_val_dataset.csv`,
  - `independent_dataset.csv`.
- **Concatenates both datasets** into a single table and standardizes column names:
  - `SEQUENCE` → `sequence`,
  - keeps the original `label` values.
- **Keeps a unified schema**:
  - `sequence`
  - `label`
- **Checks duplicated sequences** across both splits:
  - unique sequences are retained,
  - duplicates with consistent labels are collapsed,
  - sequences with conflicting labels are flagged as errors.
- **Builds metadata** from the project-wide Excel description sheet and appends QC statistics.
- **Exports curated outputs**:
  - `processed_hemolytic_dataset.csv`,
  - `detected_error_sequences.csv`,
  - `metadata.json`.

In [2]:
name_source = "HemoPI2.0"
name_task = "toxic_effect_classification"

# PATH_INPUT and PATH_EXPORT are imported from maomao.utils.constants.
# Update them in constants.py according to the required input and export paths.

- Reading raw data

In [3]:
df_cross_val = pd.read_csv(f"{PATH_INPUT}/{name_source}/cross_val_dataset.csv")

In [4]:
df_independent = pd.read_csv(f"{PATH_INPUT}/{name_source}/independent_dataset.csv")

- Concatenating dataset

In [5]:
df_hemopi2 = (
    pd.concat([df_cross_val, df_independent], ignore_index=True)
      .rename(columns={"SEQUENCE": "sequence"})
      [["sequence", "label"]]
)
df_hemopi2.shape

(1926, 2)

- Checking duplicates

In [6]:
df_remove_duplicated, df_errors, df_unique = processing_duplicated(df_hemopi2, group_seq="sequence", sort_key="label")
df_full = pd.concat([df_unique, df_remove_duplicated], axis=0)

In [7]:
df_full.shape

(1926, 2)

In [8]:
df_errors.shape

(0, 1)

- Working with metada

In [9]:
df_metada = read_metadata("../../raw_data/raw_data_description.xlsx", name_source)
dict_metadata = create_metada_with_multiple_values(df_metada)

In [10]:
dict_metadata.update({
    "number_of_raw_sequences": int(len(df_hemopi2)),
    "number_of_sequences_retained": len(df_full),
    "number_of_positive_sequences": int((df_full["label"] == 1).sum()),
    "number_of_negative_sequences": int((df_full["label"] == 0).sum()),
    "number_of_erroneous_sequences" : len(df_errors),
    "modified_sequences_included": False,
})

dict_metadata

{'type source': 'Dataset',
 'static-dynamic': 'Static',
 'license': 'No information',
 'year of publication': 2025,
 'last update date': datetime.datetime(2025, 2, 5, 0, 0),
 'download date': Timestamp('2025-08-01 00:00:00'),
 'file format': 'csv',
 'peptide property': 'hemolytic, toxic',
 'dataset information': 'Positive, Negative, Concentration constants',
 'unit of measurement': 'µM',
 'obtaining negative dataset': 'Experimentally validated, "Characteristic threshold (IC50, MIC, etc.)"',
 'repository or server': 'https://webs.iiitd.edu.in/raghava/hemopi2/index.html',
 'publication': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC11794569/',
 'number_of_raw_sequences': 1926,
 'number_of_sequences_retained': 1926,
 'number_of_positive_sequences': 891,
 'number_of_negative_sequences': 1035,
 'number_of_erroneous_sequences': 0,
 'modified_sequences_included': False}

- Exporting data

In [11]:
os.makedirs(f"{PATH_EXPORT}/{name_task}/{name_source}/", exist_ok=True)
export_json(f"{PATH_EXPORT}/{name_task}/{name_source}/metadata.json", dict_metadata)

In [12]:
df_full.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_hemolytic_dataset.csv", index=False)
df_errors.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/detected_error_sequences.csv", index=False)